In [11]:
import polars as pl
import polars.selectors as cs
import glob
from datetime import datetime

In [3]:
BHAVCOPY_PATH = "/home/parthgandhi/data/bhavcopy/NSE-Data-bank/data/"
files_list = glob.glob(BHAVCOPY_PATH + "*.csv")

In [25]:
start_date = datetime(2026, 1, 1)
end_date = datetime(2026, 4, 29)

In [26]:
data = (
    pl.scan_csv(
        files_list, with_column_names=lambda cols: [col.strip() for col in cols]
    )
    .select("SYMBOL", "SERIES", "DATE1")
    .rename({"SYMBOL": "symbol", "SERIES": "series", "DATE1": "date"})
    .with_columns(
        pl.col("date").str.strptime(pl.Date, format="%d-%b-%Y").alias("date"),
        pl.col("series").str.strip_chars().alias("series"),
        pl.col("symbol").str.strip_chars().alias("symbol"),
    )
    .filter(pl.col("series") == "EQ")
    .filter(pl.col("date").is_between(start_date, end_date, closed="both"))
    .group_by("date")
    .agg(pl.col("symbol").len().alias("symbol_count"), pl.col("symbol"))
    .sort("date")
    .collect()
)

In [27]:
data

date,symbol_count,symbol
date,u32,list[str]
2026-01-01,2383,"[""20MICRONS"", ""21STCENMGM"", … ""ZYDUSWELL""]"
2026-01-02,2380,"[""20MICRONS"", ""21STCENMGM"", … ""ZYDUSWELL""]"
2026-01-05,2389,"[""20MICRONS"", ""21STCENMGM"", … ""ZYDUSWELL""]"
2026-01-06,2387,"[""20MICRONS"", ""21STCENMGM"", … ""ZYDUSWELL""]"
2026-01-07,2388,"[""20MICRONS"", ""21STCENMGM"", … ""ZYDUSWELL""]"
…,…,…
2026-04-23,2495,"[""20MICRONS"", ""21STCENMGM"", … ""ZYDUSWELL""]"
2026-04-24,2486,"[""20MICRONS"", ""21STCENMGM"", … ""ZYDUSWELL""]"
2026-04-27,2483,"[""20MICRONS"", ""21STCENMGM"", … ""ZYDUSWELL""]"
